# Baseline
Train one model on the configs and evaluate on the held-out test split. Baselines to beat: coin flip (50%) and sign(last return).

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import yaml, numpy as np, pandas as pd
from src.data.preprocess import load_ohlcv
dc = yaml.safe_load(open('../configs/data.yaml')); dc['path'] = '../' + dc['path']
bars = load_ohlcv(dc['path'], dc['resample']); bars.tail()

In [ ]:
from src.data.dataloader import build_loaders
from src.models.factory import build_model
from src.training.trainer import Trainer, predict_loader
from src.evaluation.metrics import directional_accuracy
import torch
mc, tc = (yaml.safe_load(open(f'../configs/{n}.yaml')) for n in ('model','training'))
tc['epochs'] = 10
D = build_loaders(dc, bars); m = build_model(mc, D['n_features'], dc['seq_len'])
Trainer(m, tc, torch.device('cpu')).fit(D['train'], D['val'])
p, y = predict_loader(m, D['test'], torch.device('cpu')); print('test acc', directional_accuracy(p, y))